[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/65_roc_auc_solution.ipynb)

# Solution: ROC Curve & AUC

Reference solution.


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

def roc_auc(scores, labels):
    scores = scores.float()
    labels = labels.float()
    # sort by score, high to low
    order = torch.argsort(scores, descending=True)
    s = scores[order]
    y = labels[order]
    n_pos = y.sum().clamp_min(1.0)
    n_neg = (1 - y).sum().clamp_min(1.0)
    # cumulative TP / FP as the threshold decreases
    tp = torch.cumsum(y, dim=0)
    fp = torch.cumsum(1 - y, dim=0)
    # keep only threshold boundaries (collapse tied scores)
    distinct = torch.ones_like(s, dtype=torch.bool)
    distinct[:-1] = s[1:] != s[:-1]
    tp = tp[distinct]
    fp = fp[distinct]
    # prepend the (0, 0) origin
    tpr = torch.cat([torch.zeros(1), tp / n_pos])
    fpr = torch.cat([torch.zeros(1), fp / n_neg])
    auc = torch.trapz(tpr, fpr)
    return fpr, tpr, auc


In [ ]:
# Demo
torch.manual_seed(0)
scores = torch.rand(200)
labels = (torch.rand(200) > 0.5).long()
fpr, tpr, auc = roc_auc(scores, labels)
print('AUC:', auc.item(), '| curve points:', fpr.numel())


In [ ]:
from torch_judge import check
check('roc_auc')
